# SpaVista downstream reproduction

Run from the repository root with the environment in `requirements.txt`, `data/`, and `spavista/SpaVistav1.pth`. Paths are read from `configs/paths.yaml`. Set `RUN_TRAINING = False` to use existing results.

In [ ]:
from pathlib import Path
import glob
import json
import subprocess
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as NotebookImage

ROOT = Path.cwd()
if not (ROOT / 'configs' / 'paths.yaml').exists() and (ROOT.parent / 'configs' / 'paths.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs' / 'paths.yaml').exists(), 'Run this notebook from the SpaVista repository.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from spavista.config import load_path_config
CONFIG = ROOT / 'configs' / 'paths.yaml'
PATHS = load_path_config(CONFIG)
RUN_TRAINING = True
FIGURES = ROOT / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

def run_task(module, *options):
    command = [sys.executable, '-m', module, '--config', str(CONFIG), *options]
    print(' '.join(command))
    if RUN_TRAINING:
        env = os.environ.copy()
        env["MPLBACKEND"] = "Agg"
        subprocess.run(command, cwd=ROOT, check=True, env=env)

def show_metrics(path, columns=None):
    path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(f'{path} is missing. Run the task cell with RUN_TRAINING=True first.')
    frame = pd.read_csv(path)
    if columns is not None:
        frame = frame[columns]
    print(frame.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

def saved_figure(name):
    path = FIGURES / name
    plt.savefig(path, dpi=220, bbox_inches='tight')
    print('Saved:', path)
    plt.close()
    display(NotebookImage(filename=str(path), width=800))

def minmax(values):
    values = np.asarray(values, dtype=np.float32)
    low, high = float(np.nanmin(values)), float(np.nanmax(values))
    return np.zeros_like(values) if high <= low else (values - low) / (high - low)

## 1. Spatial gene expression prediction — cSCC, HVG

In [ ]:
SGE_FOLD = 0

In [ ]:
sge_cfg = PATHS['spatial_gene_expression']
assert sge_cfg['dataset'].lower() == 'cscc' and sge_cfg['genes_type'].upper() == 'HVG', 'This example expects cSCC-HVG results.'
assert SGE_FOLD in range(4), 'SGE_FOLD must be 0, 1, 2, or 3 for cSCC.'
SGE_RESULT = ROOT / f'results/spavista/cscc_hvg_HFAM_20epoch_fold{SGE_FOLD}'
run_task('tasks.spatial_gene_expression.run', '--fold', str(SGE_FOLD), '--device_num', '1')
sge_metrics = pd.read_csv(SGE_RESULT / 'median_performance_metrics.csv')
sge_summary = sge_metrics[['medpcc_gene', 'med_spearman_gene']].median().to_frame('Median across test slices').T
print(sge_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

In [ ]:
SGE_GENE = 'KLF6'
SGE_SLICE = 1
SGE_SLIDE = "P2_ST_rep2"
assert SGE_FOLD == 0, "P2_ST_rep2 visualization requires fold 0."
sge_cfg = PATHS['spatial_gene_expression']
assert sge_cfg['dataset'].lower() == 'cscc' and sge_cfg['genes_type'].upper() == 'HVG', 'This example expects cSCC-HVG results.'
skin = Path(sge_cfg['data_roots']['cscc'])
result = SGE_RESULT / 'npy_pred'
prefix = f'cscc_fold{SGE_FOLD}_slice{SGE_SLICE}'
true = np.load(result / f'{prefix}_true.npy')
pred = np.load(result / f'{prefix}_pred.npy')
genes = np.load(skin / 'skin_hvg_cut_1000.npy', allow_pickle=True).astype(str).tolist()
gene_index = genes.index(SGE_GENE)
cnt = pd.read_parquet(skin / 'skin_ST-cnts_171' / f'{SGE_SLIDE}_sub.parquet')
pos_file = sorted((skin / 'GSE144240_RAW').glob(f'*spot*{SGE_SLIDE}.tsv'))[0]
pos = pd.read_csv(pos_file, sep='\t')
pos['id'] = np.around(pos['x']).astype(int).astype(str) + 'x' + np.around(pos['y']).astype(int).astype(str)
spot = cnt.join(pos.set_index('id'), how='inner')
xy = spot[['pixel_x', 'pixel_y']].to_numpy(dtype=float)
assert len(xy) == len(true) == len(pred) and true.shape[1] == len(genes), 'Spot or gene order differs from the saved run.'
image_path = sorted((skin / 'GSE144240_RAW').glob(f'*{SGE_SLIDE}.jpg'))[0]
Image.MAX_IMAGE_PIXELS = None
image = Image.open(image_path)
original_width, original_height = image.size
image.draft('RGB', (2400, 2400))
image.thumbnail((2400, 2400))
image = image.convert('RGB')
xy[:, 0] *= image.width / original_width
xy[:, 1] *= image.height / original_height
fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
axes[0].imshow(image); axes[0].set_title(f'H&E — {SGE_SLIDE}')
for ax, values, title in zip(axes[1:], [true[:, gene_index], pred[:, gene_index]], ['Measured', 'SpaVista']):
    dots = ax.scatter(xy[:, 0], xy[:, 1], c=minmax(values), cmap='viridis', marker='o', s=12, vmin=0, vmax=1, linewidths=0)
    ax.set_title(f'{title} {SGE_GENE}')
    fig.colorbar(dots, ax=ax, fraction=0.045)
for ax in axes:
    ax.set_xlim(0, image.width); ax.set_ylim(image.height, 0); ax.axis('off')
saved_figure(f'sge_{SGE_SLIDE}_{SGE_GENE}.png')

## 2. High-resolution spatial expression inference — Visium HD

In [ ]:
run_task('tasks.high_resolution.run', '--target_task', 'hvg200')
run_task('tasks.high_resolution.run', '--target_task', 'marker10')
show_metrics('results/visiumhd/SpaVista_HVG200/summary_metrics_8um.csv',
             ['num_genes', 'median_gene_PCC', 'median_gene_SCC', 'median_cell_PCC'])
show_metrics('results/visiumhd/SpaVista_10markers/summary_metrics_8um.csv',
             ['num_genes', 'median_gene_PCC', 'median_gene_SCC', 'median_cell_PCC'])

In [ ]:
import torch
import scprep as scp
import os

HD_GENE = 'ERBB2'
hd_cfg = PATHS['visium_hd']
hd_root = Path(hd_cfg['base_dir'])
hd_slide = hd_cfg['test_slide']
hd_result = ROOT / 'results/visiumhd/SpaVista_10markers'
checkpoint = torch.load(hd_result / 'linear_head_ckpt/linear_head_last.pt', map_location='cpu')
hd_genes = list(checkpoint['model_config']['gene_names'])
gene_index = hd_genes.index(HD_GENE)
feature_dir = Path(hd_cfg['feature_dir'])
candidates = sorted(feature_dir.glob(f'features_{hd_slide}_test_*.npy'))
assert len(candidates) == 1, f'Expected one compatible cached feature file, found {candidates}'
features = np.load(candidates[0], mmap_mode='r')
def hd_expression_and_xy(slide):
    expression = pd.read_csv(hd_root / slide / hd_cfg['expression_file'])
    expression = expression.set_index(expression.columns[0])
    position = pd.read_parquet(hd_root / slide / 'subposition.parquet').set_index('barcode')
    meta = expression.join(position[['pxl_col_in_fullres', 'pxl_row_in_fullres']], how='left')
    assert not meta[['pxl_col_in_fullres', 'pxl_row_in_fullres']].isna().any().any()
    measured = scp.transform.log(scp.normalize.library_size_normalize(meta[hd_genes].to_numpy(dtype=np.float32)))[:, gene_index]
    xy = meta[['pxl_col_in_fullres', 'pxl_row_in_fullres']].to_numpy(dtype=np.float32)
    return measured, xy

measured_8, xy_8 = hd_expression_and_xy(hd_slide)
assert len(features) == len(measured_8)
prediction_file = FIGURES / f'{hd_slide}_{HD_GENE}_predicted_expression.npy'
inputs_time = max(candidates[0].stat().st_mtime, (hd_result / 'linear_head_ckpt/linear_head_last.pt').stat().st_mtime)
reuse_prediction = prediction_file.exists() and prediction_file.stat().st_mtime >= inputs_time
if reuse_prediction:
    prediction = np.load(prediction_file)
    reuse_prediction = prediction.shape == (len(features),)
if not reuse_prediction:
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    state = checkpoint['gene_head_state_dict']
    first_w, first_b = state['0.weight'].float().to(device), state['0.bias'].float().to(device)
    second_w = state['1.weight'][gene_index].float().to(device)
    second_b = state['1.bias'][gene_index].float().to(device)
    temp_file = prediction_file.with_suffix('.tmp')
    output = np.lib.format.open_memmap(temp_file, mode='w+', dtype=np.float32, shape=(len(features),))
    with torch.no_grad():
        for start in range(0, len(features), 2048):
            stop = min(start + 2048, len(features))
            x = torch.from_numpy(np.asarray(features[start:stop], dtype=np.float32)).to(device)
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                hidden = torch.nn.functional.linear(x, first_w, first_b)
                pred = hidden @ second_w + second_b
            output[start:stop] = pred.float().cpu().numpy()
    output.flush()
    del output
    os.replace(temp_file, prediction_file)
    prediction = np.load(prediction_file)

full_x = (float(xy_8[:, 0].min()), float(xy_8[:, 0].max()))
full_y = (float(xy_8[:, 1].min()), float(xy_8[:, 1].max()))
fig, axes = plt.subplots(1, 2, figsize=(6, 3.4), constrained_layout=True)
rows = [
    ('8 µm measured', measured_8, xy_8, 2.8),
    ('SpaVista prediction', prediction, xy_8, 2.8),
]
for ax, (label, values, xy, point_size) in zip(axes, rows):
    colors = minmax(values)
    points = ax.scatter(xy[:, 0], xy[:, 1], c=colors,
                        cmap='viridis', marker='o', s=point_size, vmin=0, vmax=1,
                        linewidths=0, rasterized=True)
    ax.set_xlim(*full_x); ax.set_ylim(full_y[1], full_y[0]); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'{label} {HD_GENE}')
fig.colorbar(points, ax=axes.tolist(), fraction=0.02, label='Min–max normalized expression')
saved_figure(f'visiumhd_{hd_slide}_{HD_GENE}.png')

## 3. HEMIT protein abundance prediction

In [ ]:
run_task('tasks.protein_abundance.run')
show_metrics('results/hemit_abundance_regression_spavista_bounded/SpaVista/test_metrics_overall.csv',
             ['median_target_pcc', 'median_target_scc', 'median_target_mae', 'best_epoch'])

In [ ]:
PROTEIN_MARKER = 'panCK_mean'
PROTEIN_TEST_INDEX = 0
from tasks.protein_abundance.run import resolve_hemit_path
protein = pd.read_csv(ROOT / 'results/hemit_abundance_regression_spavista_bounded/SpaVista/test_predictions.csv')
labels = pd.read_csv(ROOT / PATHS['hemit']['labels_csv'])
test = labels.loc[labels['split'].eq('test')].dropna(subset=['panCK_mean', 'panCK_pos_area', 'CD3_mean', 'CD3_pos_area', 'he_path']).reset_index(drop=True)
data_root = Path(PATHS['hemit']['root'])
test['he_path'] = test['he_path'].map(lambda p: resolve_hemit_path(p, str(data_root), 'test', 'input'))
test['label_path'] = test['label_path'].map(lambda p: resolve_hemit_path(p, str(data_root), 'test', 'label'))
test = test.loc[test['he_path'].map(lambda p: Path(p).exists())].reset_index(drop=True)
assert len(test) == len(protein), 'Test sample order or filtering differs from the saved predictions.'
row = test.iloc[PROTEIN_TEST_INDEX]
prediction = protein.iloc[PROTEIN_TEST_INDEX]
assert np.isclose(float(row[PROTEIN_MARKER]), float(prediction['true_' + PROTEIN_MARKER]), atol=1e-5)
he = Image.open(row['he_path']).convert('RGB')
label = np.asarray(Image.open(row['label_path']))
panck = label[..., 0] if label.ndim == 3 else label
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.9), constrained_layout=True)
axes[0].imshow(he); axes[0].set_title('H&E input')
axes[1].imshow(panck, cmap='magma'); axes[1].set_title('Measured panCK fluorescence')
for ax in axes: ax.axis('off')
fig.suptitle(f"panCK mean: measured {prediction['true_panCK_mean']:.4f}  |  SpaVista {prediction['pred_panCK_mean']:.4f}")
saved_figure(f'protein_test_{PROTEIN_TEST_INDEX}_panCK.png')

## 4. H&E-to-mIF image reconstruction

In [ ]:
run_task('tasks.mif_generation.run')
show_metrics('results/hemit_mif_spavista_bs6_10epoch_mean_ssim/SpaVista/test_metrics_overall.csv',
             ['mean_pixel_pcc', 'mean_global_ssim', 'mean_pixel_mae', 'mean_psnr', 'best_epoch'])

In [ ]:
from types import SimpleNamespace
from tasks.mif_generation import run as mifmod

MIF_TILE = '[8942,53607]_patch_6_7'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
args = SimpleNamespace(spavista_ckpt=PATHS['checkpoint'], spavista_arch='mamba_vision_base',
                       strict_load=False, num_feature_levels=4, image_size=224,
                       normal=True, amp='fp16' if device.type == 'cuda' else 'none')
dataset = mifmod.HEMITImageReconstructionDataset(
    csv_path=str(ROOT / PATHS['hemit']['labels_csv']), split='test',
    out_channels=['panCK', 'CD3'], data_root=str(PATHS['hemit']['root']), verbose=False)
matches = dataset.df.index[dataset.df['tile_id'].eq(MIF_TILE)].tolist()
assert len(matches) == 1, f'Expected one test tile named {MIF_TILE}, found {len(matches)}'
he_tensor, true_tensor, _ = dataset[matches[0]]
encoder, _, model_name = mifmod.build_encoder(args, 'spavista')
spatial_encoder = mifmod.MultiStageFeatureExtractor(encoder, model_name, num_feature_levels=4)
shapes = mifmod.infer_feature_shapes(spatial_encoder, args, device)
decoder = mifmod.MultiStageUNetDecoder(
    in_channels_list=[shape[0] for shape in shapes], out_channels=2,
    decoder_dim=128, image_size=224, num_post_blocks=2, norm='gn')
model = mifmod.FrozenMultiStageEncoderUNet(spatial_encoder, decoder)
mifmod.load_best_decoder(model, str(ROOT / 'results/hemit_mif_spavista_bs6_10epoch_mean_ssim/SpaVista'), device)
with torch.no_grad():
    with mifmod.autocast_context(args, device):
        pred_tensor = model(he_tensor.unsqueeze(0).to(device))[0].float().cpu().clamp(0, 1)
true_tensor = true_tensor.clamp(0, 1)
he_image = mifmod.tensor_to_numpy_img(mifmod.denormalize_he(he_tensor))
true_values, pred_values = true_tensor.numpy(), pred_tensor.numpy()
scales = np.maximum(np.percentile(true_values.reshape(2, -1), 99.8, axis=1), 1e-6)
def display_mif(values):
    enhanced = np.clip(values / scales[:, None, None], 0, 1) ** 0.65
    rgb = np.zeros((224, 224, 3), dtype=np.float32)
    rgb[..., :2] = enhanced.transpose(1, 2, 0)
    return rgb
true_display, pred_display = display_mif(true_values), display_mif(pred_values)
channel_pcc = [np.corrcoef(true_values[i].ravel(), pred_values[i].ravel())[0, 1] for i in range(2)]
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6), constrained_layout=True)
for ax, img, title in zip(axes, [he_image, true_display, pred_display],
                          ['H&E', f'True mIF — {MIF_TILE}', f'SpaVista (mean PCC={np.mean(channel_pcc):.3f})']):
    ax.imshow(img); ax.set_title(title, fontsize=8, pad=3); ax.axis('off')
saved_figure('mif_8942_53607_patch_6_7.png')
del model, decoder, spatial_encoder, encoder
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 5. Tumor lesion detection — TRIST

In [ ]:
run_task('tasks.tumor_detection.run')
show_metrics('results/tumor_detection_linear_probe/SpaVista/metrics.csv',
             ['accuracy', 'balanced_accuracy', 'f1', 'auroc', 'auprc'])

In [ ]:
from tasks.tumor_detection.run import STNetDataClassifyLazy

TUMOR_SECTION = '23268_D1'
tumor_cfg = PATHS['tumor_detection']
test_folders = str(tumor_cfg['test_folders']).split(',')
dataset = STNetDataClassifyLazy(str(tumor_cfg['root_dir']), test_folders, train=False, train_aug=False, cache_images=False, verbose=False)
saved = pd.read_csv(ROOT / 'results/tumor_detection_linear_probe/SpaVista/test_predictions.csv')
assert len(dataset.samples) == len(saved)
assert np.array_equal(np.array([sample['label'] for sample in dataset.samples]), saved['y_true'].to_numpy()), 'Saved prediction order does not match dataset order.'
selected = [i for i, sample in enumerate(dataset.samples) if TUMOR_SECTION in Path(sample['tif_path']).stem]
assert selected, f'No test spots found on {TUMOR_SECTION}'
sample = dataset.samples[selected[0]]
xy = np.array([[dataset.samples[i]['x'], dataset.samples[i]['y']] for i in selected])
image = Image.open(sample['tif_path'])
original_width, original_height = image.size
image.thumbnail((2400, 2400))
image = image.convert('RGB')
xy = xy.astype(float)
xy[:, 0] *= image.width / original_width
xy[:, 1] *= image.height / original_height
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2), constrained_layout=True)
axes[0].imshow(image); axes[0].set_title('H&E')
for ax, values, title in [
    (axes[1], saved.iloc[selected]['y_true'].to_numpy(), 'Ground truth'),
    (axes[2], saved.iloc[selected]['y_pred_for_metrics'].to_numpy(), 'SpaVista'),
]:
    colors = np.where(values.astype(int) == 1, '#f3cb55', '#b9bbbd')
    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=55, linewidths=0)
    ax.set_title(title)
for ax in axes:
    ax.set_xlim(0, image.width); ax.set_ylim(image.height, 0); ax.set_aspect('equal'); ax.axis('off')
saved_figure(f'tumor_{TUMOR_SECTION}.png')

## Figure files

Figures are saved in `notebooks/figures/`.